# Step 4 — PlanTUS: Interactive Transducer Placement

Runs PlanTUS to interactively place a transducer on the scalp surface. A SimNIBS
head mesh is converted to wb_view surfaces with placement metric maps; clicking a
scalp vertex triggers optimised placement computation. The result is exported as a
BrainSight-compatible coordinate file.

**Typical interaction time:** 5–15 min per session.

[← Step 3 — Inverse Registration](step03_inverse_registration.ipynb) &nbsp;|&nbsp; **Step 4** &nbsp;|&nbsp; Step 5 →

In [ ]:
# Reference card — displays the pipeline overview and all option tables.
# Run this cell once to see settings options before editing the Settings cell below.
exec(open(__import__('pathlib').Path(__vsc_ipynb_file__).parent / 'z_references/step4_reference.py').read())


In [ ]:

# -----------------------------------------------------------------------
# Settings — edit here before running
# -----------------------------------------------------------------------
# Safe to Run All. Every stage below either does its work or reports that it is
# skipping and why:
#   4b  placement — skipped when a vertex already exists (REUSE_PLACEMENT)
#   4c  export    — always runs; picks the vertex via VTX
#   4d  L+R merge — self-skips unless both sides exist
#
# sub-S02 aMCC. No PlanTUS folder exists for this target yet (the step-3 masks
# are there, the PlanTUS output was removed), so 4b builds the surfaces and
# metric maps from scratch and opens wb_view for a placement.
#
# ⚠ After a NEW placement, re-run step 5 with REUSE_FILES = False.
#   The step-5 prefix comes from the trajectory's target *name*, which does not
#   change, so REUSE_FILES = True would silently reuse the domain and acoustic
#   files built from the OLD trajectory.

SITE_YAML = "../config/sites/site_UMD_AK.yaml"
SUB_ID    = "sub-S02"
SUB_LIST_DIR = ""   # subject folder under data_root; "" = the site YAML's sub_list_dir

# ── Target ────────────────────────────────────────────────
# Both splits are non-empty here (L 997 voxels, R 3506), so 4d can merge them
# once both sides have a placement. Run "_L" first, then "_R".
TARGET_NAME       = "aMCC_NeuroSynthTopic112"     # target label (matches mask label from step 3)
TARGET_SIDE       = "_L"                          # "_L" first, then "_R"; "" for bilateral

# ⚠ A pad is REQUIRED for this target — measured on sub-S02, over the scalp
# vertices that pass avoidance and aim ≤ 10°:
#
#            aim-OK   inside 50–120 mm   below 50 mm        above 120 mm
#   aMCC_L     824           0           127 (38.6–40.1)        697
#   aMCC_R     818           0           139 (40.4–42.3)        679
#
# The scalp points that aim at aMCC are either straight above it (too shallow
# for the DPX-500) or on the far side of the head (too deep). Nothing lands in
# the focal window unaided, which is not true of the hippocampus (~103 mm).
#
# 20 mm lifts the near group to ~58–62 mm, i.e. into the 60–120 mm range the
# vendor actually calibrated. 10 mm would only clear the 50 mm hardware floor
# and leave the setting below any calibration data.
ADDITIONAL_OFFSET = 20.0                          # gel/pad offset (mm)

# ── Placement reuse / vertex choice ───────────────────────
# REUSE_PLACEMENT=True keeps 4b from opening wb_view when a placement already
# exists — this is what makes Run All work, since 4b is the one interactive
# stage. Set False to place a new vertex (requires clicking in wb_view).
REUSE_PLACEMENT = False

# Which vertex 4c exports. Leave as None: that uses the newest placement, which
# is also the only one when a single placement exists. Pin an index only to
# override, e.g. to re-export an earlier placement for comparison.
VTX = None

# ── Run options ───────────────────────────────────────────
DRY_RUN = False   # True = validate inputs only; no files written

# True  — use pynput mouse listener (requires macOS Accessibility permission)
# False — parse wb_view FINER log directly; no Accessibility permission needed
USE_PYNPUT = True

# Fraction of max intersection to define top-candidate pool for auto vertex selection.
# Higher → fewer candidates (stricter intersection filter, but may force high angle).
# Lower  → more candidates (more angle/distance weight in tiebreak).
# Only used when a new placement is made (REUSE_PLACEMENT=False).
TOP_PCT = 0.5

# Tiebreak weighting inside that pool: (aim angle, distance, skin-skull angle).
# Each term is normalised to [0,1] across the pool and summed; lowest wins.
#   aim°  — scalp normal vs scalp→target. Often near-constant within a pool
#           (sub-S02: the four candidates span 0.9°), so it carries little.
#   dist  — scalp to ROI centre of gravity.
#   skl°  — obliquity on bone. This is the term that tracks transmission:
#           on sub-S01 derating was 0.0478 at 3.8° vs 0.0413 at 26.7°.
# (1, 1, 0) reproduces the pre-2026-08 behaviour, which ignored skl°.
VTX_WEIGHTS = (1.0, 1.0, 1.0, 1.0)   # inter, aim°, dist, skl° — rank-based

# Radius of the blob drawn at EVERY candidate in wb_view. Scalp vertices sit
# ~1.8 mm apart, so a bare candidate is a dot you can neither see nor hit with
# the mouse; 3 mm is ~7 vertices, about 5 mm across. Blobs are painted in rank
# order onto unclaimed vertices, so a better rank always wins an overlap and no
# candidate is buried.
#
# The marker layer carries each candidate's RANK, so wb_view's hover tooltip
# ("Top Enabled Layer") reads 1 for the suggestion, 2 for the runner-up, and so
# on. For the numbers behind a rank, type ?N at the placement prompt.
MARK_RADIUS = 3.0

# ── BrainSight coordinate system ───────────────────────────
# Options: "Brainsight", "NIfTI:Q:Scanner", "NIfTI:S:Scanner"
COORDINATE_SYSTEM = "NIfTI:S:Scanner"

print("Settings loaded.")
print(f"  Subject:          {SUB_ID}")
print(f"  Target:           {TARGET_NAME}{TARGET_SIDE}")
print(f"  DRY_RUN:          {DRY_RUN}")
print(f"  REUSE_PLACEMENT:  {REUSE_PLACEMENT}")
print(f"  VTX:              {VTX if VTX is not None else 'newest'}")
print(f"  TOP_PCT:          {TOP_PCT}")
print(f"  VTX_WEIGHTS:      aim×{VTX_WEIGHTS[0]:g} + dist×{VTX_WEIGHTS[1]:g}"
      f" + skin-skull×{VTX_WEIGHTS[2]:g}")
print(f"  MARK_RADIUS:      {MARK_RADIUS} mm")


In [ ]:

# -----------------------------------------------------------------------
# Imports & path setup
# -----------------------------------------------------------------------

# ── Imports (stdlib) ──────────────────────────────────────
import sys
from pathlib import Path

NB_DIR = Path(__vsc_ipynb_file__).parent  # notebook directory (cwd-independent anchor)

# ── Config — Python path ──────────────────────────────────
# Make src/ importable; one level up from notebook/
SRC_DIR = str((NB_DIR / "../src").resolve())
if SRC_DIR not in sys.path:sys.path.insert(0, SRC_DIR)

# ── Paths — resolve relative to notebook file (cwd-independent) ───────
SITE_YAML = str((NB_DIR / SITE_YAML).resolve())

# ── src / utils ───────────────────────────────────────────
from utils import (
    load_site_config,
    resolve_data_dir,
    resolve_sub_dir,
    normalise_sub_id,
    load_transducer_config,
    transducer_params,
    setup_environment,
    run_plantus,
    list_plantus_vertices,
    write_brainsight_for_vtx,
    find_plantus_target_folder,
    merge_brainsight_files,
)

print("Imports complete.")


In [ ]:

# -----------------------------------------------------------------------
# Step 4-0 — Load config, resolve paths, set up environment
# Validates SimNIBS m2m directory; configures PATH; loads transducer params.
# -----------------------------------------------------------------------

cfg = load_site_config(SITE_YAML)
if SUB_LIST_DIR:
    cfg["sub_list_dir"] = str(Path(SUB_LIST_DIR).expanduser())

data_dir                 = resolve_data_dir(cfg)
sub_id_full, sub_id_bare = normalise_sub_id(SUB_ID)
sub_dir                  = resolve_sub_dir(data_dir, sub_id_bare, sub_id_full)
m2m_dir                  = sub_dir / f"m2m_{sub_id_full}"

if not m2m_dir.exists():
    raise FileNotFoundError(f"SimNIBS m2m directory not found: {m2m_dir}")

print(f"Subject:  {sub_id_full}")
print(f"Data dir: {data_dir}")
print(f"sub_dir:  {sub_dir}")
print(f"m2m_dir:  {m2m_dir}")

setup_environment(cfg)

tcfg = load_transducer_config(cfg, SITE_YAML)
tp   = transducer_params(tcfg)

print(f"\nTransducer:       {tcfg.get('name', 'unknown')}")
print(f"Focal depth (mm): {tp.get('focal_depth_mm')}")


In [ ]:

# -----------------------------------------------------------------------
# Step 4b — PlanTUS interactive placement
# 1. prepscene: mesh → surfaces + metric maps + scene file → wb_view opens.
# 2. Click scalp vertex → "yes" runs placement; "no" repeats selection.
#
# With REUSE_PLACEMENT=True this skips (and says so) when a vertex folder
# already exists, so the notebook can be run top-to-bottom without blocking on
# wb_view. Set REUSE_PLACEMENT=False to place a new vertex.
# DRY_RUN=True validates inputs only; no files written.
# -----------------------------------------------------------------------

run_plantus(
    sub_id_full=sub_id_full,
    sub_id_bare=sub_id_bare,
    m2m_dir=m2m_dir,
    target_name=TARGET_NAME,
    target_side=TARGET_SIDE,
    tp=tp,
    additional_offset=ADDITIONAL_OFFSET,
    dry_run=DRY_RUN,
    use_pynput=USE_PYNPUT,
    top_pct=TOP_PCT,
    reuse_placement=REUSE_PLACEMENT,
    weights=VTX_WEIGHTS,
    mark_radius_mm=MARK_RADIUS,
)


### Step 4b-alt — Launch `PlanTUS_wrapper.py` as a subprocess

Use the cell below **instead of** Step 4b above when the notebook kernel is **not** `simnibs_python`.  
The wrapper receives all settings from the cells above — no duplication required.

In [ ]:
# # -----------------------------------------------------------------------
# # Step 4b-alt — Launch PlanTUS_wrapper.py as a subprocess
# # All settings are forwarded from the Settings cell above.
# # -----------------------------------------------------------------------
# import subprocess

# _simnibs_py  = cfg.get("simnibs_python", "simnibs_python")
# _wrapper     = str((Path(SITE_YAML).parent.parent.parent / "PlanTUS" / "PlanTUS_wrapper.py").resolve())

# _cmd = [
#     str(_simnibs_py),
#     _wrapper,
#     "--site",              SITE_YAML,
#     "--sub",               SUB_ID,
#     "--target",            TARGET_NAME,
#     "--side",              TARGET_SIDE,
#     "--additional-offset", str(ADDITIONAL_OFFSET),
# ]
# if DRY_RUN:
#     _cmd.append("--dry-run")

# print("Command:\n  " + " \\\n    ".join(_cmd))
# subprocess.run(_cmd, check=True)

In [ ]:
# -----------------------------------------------------------------------
# Step 4c — Export BrainSight-compatible coordinate file
# Output: <PlanTUS target folder>/{sub}_T1w_{target}{side}_target_coordinates_Brainsight_PlanTUS.txt
#
# This is the file step 5 reads. VTX=None (the default) uses the newest vertex —
# which is also the only one when a single placement exists. Pin VTX only to
# override that, e.g. to compare two placements of the same target.
# -----------------------------------------------------------------------

list_plantus_vertices(find_plantus_target_folder(
    m2m_dir, sub_id_full, TARGET_NAME, TARGET_SIDE))

write_brainsight_for_vtx(
    m2m_dir=m2m_dir,
    sub_id_full=sub_id_full,
    target_name=TARGET_NAME,
    target_side=TARGET_SIDE,
    coordinate_system=COORDINATE_SYSTEM,
    vtx=VTX,
)


In [ ]:
# -----------------------------------------------------------------------
# Step 4d — Merge L+R BrainSight files into a single combined file
# -----------------------------------------------------------------------
#
# Run this cell after running Step 4c for BOTH _L and _R sides.
# Looks for individual per-side BrainSight files and merges them into:
#   <m2m_dir/PlanTUS/combined>/
#     {sub_id}_T1w_{TARGET_NAME}_target_coordinates_Brainsight_PlanTUS.txt
#
# If only one side has been run, prints a warning and skips.

_side_paths = {}
for _side in ("_L", "_R"):
    try:
        _folder = find_plantus_target_folder(m2m_dir, sub_id_full, TARGET_NAME, _side)
        _label = f"{sub_id_full}_T1w_{TARGET_NAME}{_side}"
        _cand = _folder / f"{_label}_target_coordinates_Brainsight_PlanTUS.txt"
        if _cand.exists():
            _side_paths[_side] = _cand
    except SystemExit:
        pass  # folder doesn't exist yet

if len(_side_paths) < 2:
    print(f"WARNING: found {list(_side_paths.keys())} — both _L and _R required for merge.")
    print("Run Step 4c for the missing side first, then re-run this cell.")
else:
    _combined_dir = m2m_dir / "PlanTUS" / "combined"
    _combined_dir.mkdir(parents=True, exist_ok=True)
    _combined_path = _combined_dir / f"{sub_id_full}_T1w_{TARGET_NAME}_target_coordinates_Brainsight_PlanTUS.txt"
    merge_brainsight_files(
        in_paths=[_side_paths["_L"], _side_paths["_R"]],
        out_path=_combined_path,
    )
    print(f"  Contains L (_target + _entry) and R (_target + _entry) rows.")
    print(f"  Ready to import into BrainSight.")
